# **DURABILITY — $\lambda$ MAPS**

Scatter of every design point at a chosen time step: one design variable on $x$, another on $y$,
$\lambda_i$ on the color bar — one **separate** figure per $\lambda_i$ ($i=1,\dots,4$), each saved
on its own so they can be arranged freely in Overleaf. No title on the figures themselves — that
belongs in the LaTeX caption.

Loads `dataset_unique_<split>_<tag>.pkl` from
[`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) — **not** `dataset_full`: `dataset_full`
has one row per *latent* draw (thousands per design point, all sharing the same design point and
$\lambda$), so plotting it would just redraw the same dot on top of itself thousands of times for
no visual gain. `dataset_unique` already has exactly one row per design point.

The durability problem has **three** design variables (`fck`, `rh`, `cov`), but a scatter only has
room for two spatial axes plus color — pick the pair with `x_var`/`y_var` below; the third variable
is simply not shown (it still varies across the points plotted, it's just not encoded visually).

## 1. Libraries

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False
                    })
import numpy as np
import pandas as pd

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import dill

## 2. Config

`n_latent_samples`, `times`, `installation_year`, `cement_type` and `exposure_conditions` must
match [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) — together they name the file being
loaded. Pick the time step with `time_index` (its position in `times`) rather than typing the float
directly, to avoid a mismatch against the saved filename.

`x_var`/`y_var` choose which 2 of the 3 design variables (`'fck'`, `'rh'`, `'cov'`) go on the
scatter's axes.

`xlim`/`ylim` and `lambda_vlim` are `None` by default (matplotlib auto-scales each figure to its
own data). Set them explicitly to force the same axis range / color range across figures — e.g.
across different `time_index` values, so the panels are visually comparable in the paper.

In [ ]:
n_latent_samples     = 100000   # must match stage 1 — it names the file being loaded
times                = np.linspace(0, 150, 10, endpoint=True)  # must match stage 1
installation_year    = 1990     # must match stage 1
cement_type          = 3        # must match stage 1
exposure_conditions  = 2        # must match stage 1

split       = 'train'   # 'train' or 'val'
time_index  = 5          # index into `times` — change this to look at a different time step
time_step   = times[time_index]

x_var = 'fck'   # 'fck', 'rh' or 'cov' — plotted on the x axis
y_var = 'rh'    # 'fck', 'rh' or 'cov' — plotted on the y axis

fig_size   = (5, 4)      # size of each individual figure, in inches
fig_format = 'png'       # format each figure is saved in ('pdf', 'png', ...)
fig_dpi    = 300         # resolution the figure is saved at (dots per inch)

label_fontsize = 14   # font size of the axis labels and colorbar label
tick_fontsize  = 12   # font size of the tick numbers, on both axes and the colorbar

xlim = None   # e.g. (20, 50) to fix the x axis; None = auto-scaled to the data
ylim = None   # e.g. (20, 80) to fix the y axis; None = auto-scaled to the data

# Per-lambda colorbar limits as (vmin, vmax). None = auto-scaled to that lambda's own range.
lambda_vlim = {
                'lambda 1': None,
                'lambda 2': None,
                'lambda 3': None,
                'lambda 4': None,
              }

print(f"Plotting t = {time_step:.2f} years ({split} split), {x_var} vs {y_var}")

## 3. Load the dataset

In [ ]:
tag      = f'{time_step}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}'
pkl_name = f'{n_latent_samples}_dataset_unique_{split}_{tag}'

with open(f'{pkl_name}.pkl', 'rb') as f:
    df = dill.load(f)

print(f"{len(df)} design points loaded")
df.head()

## 4. $\lambda$ maps

One figure per $\lambda_i$, each with its own color scale by default (the four $\lambda$'s live on
very different ranges) — saved as `<pkl_name>_lambda_<i>.<fig_format>`, i.e. the same name as the
`.pkl` this notebook loaded, so the figure and the data it came from are easy to match up later.

In [ ]:
var_labels = {'fck': '$f_{ck}$ (MPa)', 'rh': '$RH$ (%)', 'cov': '$c$ (mm)'}

lambda_cols = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']
figs = {}

for col in lambda_cols:
    mask       = df[col].notna()
    vmin, vmax = lambda_vlim.get(col) or (None, None)
    idx        = col.split(' ')[1]

    fig, ax = plt.subplots(figsize=fig_size)
    sc = ax.scatter(df.loc[mask, x_var], df.loc[mask, y_var], c=df.loc[mask, col],
                     cmap='coolwarm', s=25, edgecolor='0.3', linewidth=0.3, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(f'$\\lambda_{{{idx}}}$', fontsize=label_fontsize)
    cbar.ax.tick_params(labelsize=tick_fontsize)
    ax.set_xlabel(var_labels[x_var], fontsize=label_fontsize)
    ax.set_ylabel(var_labels[y_var], fontsize=label_fontsize)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    fig.savefig(f'{pkl_name}_lambda_{idx}.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
    figs[col] = fig
    plt.show()

## 5. $\lambda$ maps — 3D (all three design variables at once)

Since the durability problem actually has 3 design variables, this puts `fck`, `rh` and `cov` on
the 3 spatial axes together and keeps $\lambda_i$ on the color bar — one figure per $\lambda_i$,
same conventions as section 4 (no title, configurable size/fonts/dpi, saved next to the `.pkl`).
`elev`/`azim` set the camera angle.

In [ ]:
fig_size_3d = (7, 6)   # 3D plots need more room than the 2D ones for the extra axis + colorbar
elev = 20   # camera elevation angle, in degrees
azim = -60  # camera azimuth angle, in degrees

figs_3d = {}

for col in lambda_cols:
    mask       = df[col].notna()
    vmin, vmax = lambda_vlim.get(col) or (None, None)
    idx        = col.split(' ')[1]

    fig = plt.figure(figsize=fig_size_3d)
    ax  = fig.add_subplot(projection='3d')
    sc = ax.scatter(df.loc[mask, 'fck'], df.loc[mask, 'rh'], df.loc[mask, 'cov'], c=df.loc[mask, col],
                     cmap='coolwarm', s=20, edgecolor='0.3', linewidth=0.2, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.15)
    cbar.set_label(f'$\\lambda_{{{idx}}}$', fontsize=label_fontsize)
    cbar.ax.tick_params(labelsize=tick_fontsize)
    ax.set_xlabel(var_labels['fck'], fontsize=label_fontsize, labelpad=15)
    ax.set_ylabel(var_labels['rh'], fontsize=label_fontsize, labelpad=15)
    ax.set_zlabel(var_labels['cov'], fontsize=label_fontsize, labelpad=15)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
    ax.view_init(elev=elev, azim=azim)

    fig.savefig(f'{pkl_name}_lambda_{idx}_3d.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
    figs_3d[col] = fig
    plt.show()